# 03 SHAP Analysis

This notebook prepares a tree-based regressor for feature attribution analysis. If `shap` is available, SHAP values can be computed directly; otherwise, the notebook falls back to model feature importances.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)


/root/大创/perovskite-bandgap-ml-correction


In [2]:
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor

from src.data_pipeline import load_training_set, sanitize_columns, build_feature_matrix

df = sanitize_columns(load_training_set(repo_root / 'data' / 'training_set_257.csv'))
df_nm = df[df['E_g_Exp'] > 0.05].reset_index(drop=True)
drop_cols = ['Formula', 'E_g_Exp', 'Source', 'Priority', 'pretty_formula', 'Delta_E_g', 'is_metal_exp', 'target_delta']
X, feature_cols = build_feature_matrix(df_nm, drop_cols)
y = df_nm['E_g_Exp'].values

reg = ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=42)
reg.fit(X, y)
print(X.shape, len(feature_cols))


(115, 285) 285


In [3]:
importance_df = pd.DataFrame({'feature': feature_cols, 'importance': reg.feature_importances_})
importance_df = importance_df.sort_values('importance', ascending=False).reset_index(drop=True)
importance_df.head(20)


,feature,importance
0,band_gap,0.422946
1,Sn fraction,0.102293
2,is_metal_gga,0.058825
3,minimum EN difference,0.026457
4,formation_energy_per_atom,0.016778
5,MagpieData maximum GSbandgap,0.011173
6,MagpieData mean GSmagmom,0.010212
7,MagpieData avg_dev GSmagmom,0.009479
8,avg ionic char,0.008011
9,MagpieData range SpaceGroupNumber,0.007919


In [4]:
try:
    import shap
    explainer = shap.TreeExplainer(reg)
    sample = X.iloc[:50]
    shap_values = explainer.shap_values(sample)
    shap_df = pd.DataFrame(shap_values, columns=sample.columns)
    shap_df.abs().mean().sort_values(ascending=False).head(20)
except Exception as exc:
    print('SHAP is unavailable in the current environment:', exc)
    importance_df.head(20)
